### This notebook simulates elastic waveforms with Deepwave for randomly generated velocity models, one source per model. Please adjust the number of velocity models as needed. 

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = 'cuda'
import gc
import torch
import numpy as np
import gstools as gs
import deepwave

from scipy.interpolate import interp1d

ModuleNotFoundError: No module named 'gstools'

In [ ]:
def generate_vs(nxin, nyin, vsbg, len_scale=[64, 8], sigma=20, seed=-1):
    rf = generate_2DRF(nxin, nyin, len_scale=len_scale, sigma=sigma, nu=1.5, seed=seed)
    vs = perturbation(vsbg, rf)
    vs[vs<500] = 500
    return vs

def brocher_rho_from_vp(vp):
    '''
    Calculate rho from vp using Brocher, 2005
    valid for vp between 1.5 and 8.5 km/s
    vp: m/s
    rho: kg/m^3
    '''
    vp = vp / 1000  # to km/s
    rho = 1.6612*vp - 0.4721*vp**2 + 0.0671*vp**3 - 0.0043*vp**4 + 0.000106*vp**5  # g/cc
    rho = rho * 1000  # to kg/m^3
    return rho

def brocher_vp_from_vs(vs):
    '''
    Calculate vp from vs using Brocher, 2005
    valid for vs between 0 and 4.5 km/s
    vs: m/s
    vp: m/s
    '''
    vs = vs / 1000  # to km/s
    vp = 0.9409 + 2.0947*vs - 0.8206*vs**2 + 0.2683*vs**3 - 0.0251*vs**4
    vp = vp * 1000  # to m/s
    vp[vp>12000] = 12000
    return vp

def generate_2DRF(nx, ny, len_scale=32, sigma=10, nu=1.5, seed=-1):
    '''
    Generate a 2D Matern random field
    Check https://geostat-framework.readthedocs.io/projects/gstools/en/stable/ for more details
    len_scale: correlation length in number of grid points
    sigma: standard deviation in percent
    seed: if needed
    ''' 
    pad = len_scale  # eliminate the periodicity induced by the Fourier method
    Nx = nx + 2 * pad
    Ny = ny + 2 * pad
    x, y = np.arange(Nx, dtype=float), np.arange(Ny, dtype=float)
    model = gs.Matern(dim=2, var=sigma**2, len_scale=len_scale, nu=nu)
    if seed >= 0:
        srf = gs.SRF(model, seed=seed, generator='Fourier', period=[nx, ny])  # default RandMeth produces artifacts
    else:
        srf = gs.SRF(model, generator='Fourier', period=[nx, ny])

    dV = srf((x, y), mesh_type='structured')
    dV = dV[pad:-pad, pad:-pad]
    return dV

def perturbation(bg, rf):
    '''
    bg: background model, (nx, ny)
    rf: random field in percent, (nx, ny)
    '''
    out = bg * (1 + (rf / 100))
    return out

def generate_sigma(low=10, high=20):
    '''
    generate sigma from uniform(low, high)
    '''
    sigma = np.random.uniform(low, high)    
    return sigma

def generate_len_scale(grid_spacing, low_physical=1e3, high_physical=10e3):
    physical_len_scale = np.random.uniform(low_physical, high_physical)
    grid_len_scale = int(physical_len_scale/grid_spacing)
    return grid_len_scale

In [ ]:
# finest grid spacing and time step allowed by the physics
vmin = 500
fmax = 0.5
ngppsw = 8  # number of grid points per shortest wavelength
h_phy = vmin / (fmax * ngppsw)  # m

vmax = 12000
dt_phy = h_phy / vmax * 0.1  # s, CFL condition with a safety factor of 0.1

In [ ]:
xrange = 85e3 
zrange = 20e3
h = 125.0  # spacing in m
assert h <= h_phy
nx = int(1.5+xrange/h)
nz = int(1.5+zrange/h)
xcoor = torch.arange(nx) * h
zcoor = torch.arange(nz) * h
pml_width = 20e3  # if too small, there will be reflection artifacts from boundaries
pml_width_in_cell = int(pml_width // h)

In [ ]:
# 1D background Vs: average of CVM-S4.26 profiles
z_1d, vs_1d = np.loadtxt("vsbg_1d.txt", unpack=True)
f = interp1d(z_1d, vs_1d, kind="linear")
vsbg = np.tile(f(zcoor).reshape(1, -1), (nx, 1))

In [ ]:
# acquisition
recxidx = np.arange(nx)[::2][1:-1]
recxs = xcoor[recxidx]
n_shots = 1
n_sources_per_shot = 1
source_depth = 2  # in number of cells
n_receivers_per_shot = len(recxidx)
receiver_depth = 2

receiver_locations = torch.zeros(n_shots, n_receivers_per_shot, 2, dtype=torch.long, device=device)
receiver_locations[..., 1] = receiver_depth
receiver_locations[:, :, 0] = torch.tensor(recxidx).repeat(n_shots, 1)

# source time function
freq = 0.3  # center frequency in Hz
peak_time = 1.5 / freq
end_time_in_seconds = 50.0
dt = 1e-3
assert dt <= dt_phy
nt = int(1.5+(end_time_in_seconds+peak_time)/dt)
source_amplitudes = deepwave.wavelets.ricker(freq, nt, dt, peak_time).repeat(n_shots, n_sources_per_shot, 1).to(device)

# downsampling in time
dt_down = 0.1  # in s
sampling_interval_in_time_steps = int(dt_down/dt)
assert sampling_interval_in_time_steps == dt_down / dt
i0 = round(peak_time/dt_down)  # record from the peak time of the source

In [ ]:
nvel = 2  # number of velocity models
save_path = "../data_raw/"
os.makedirs(save_path, exist_ok=True)

In [ ]:
i = 0
while i < nvel:
    srcxidx = np.random.choice(np.arange(nx)[1:-1], n_shots, replace=False)
    srcxs = xcoor[srcxidx]
    source_locations = torch.zeros(n_shots, n_sources_per_shot, 2, dtype=torch.long, device=device)
    source_locations[..., 1] = source_depth
    source_locations[:, 0, 0] = torch.tensor(srcxidx)

    vs = torch.from_numpy(generate_vs(nx, nz, vsbg, len_scale=generate_len_scale(grid_spacing=h), sigma=generate_sigma())).float().to(device)
    vp = brocher_vp_from_vs(vs)
    rho = brocher_rho_from_vp(vp)
    with torch.no_grad():
        propagator = deepwave.Elastic(
            *deepwave.common.vpvsrho_to_lambmubuoyancy(vp, vs, rho),
            grid_spacing=h, 
            )
        out = propagator(
            dt=dt, 
            source_amplitudes_x=source_amplitudes,  # sources oriented in the second spatial dimension (z)
            source_locations_x=source_locations,
            receiver_locations_x=receiver_locations,
            receiver_locations_y=receiver_locations,
            accuracy=4,
            pml_width=[pml_width_in_cell, pml_width_in_cell, 0, pml_width_in_cell],  # in number of cells, free surface at z=0
            pml_freq=freq,
            )
        
    vz = out[-1].cpu().numpy()[..., ::sampling_interval_in_time_steps][..., i0:]  # (nsrc, nrec, nt)
    vx = out[-2].cpu().numpy()[..., ::sampling_interval_in_time_steps][..., i0:]  # (nsrc, nrec, nt)

    if np.isnan(vz).sum() == 0 and np.isnan(vx).sum() == 0:
        np.save(save_path+"vz{}.npy".format(i), vz.astype(np.float32))
        np.save(save_path+"vx{}.npy".format(i), vx.astype(np.float32))
        np.save(save_path+"vs{}.npy".format(i), vs.cpu().numpy().astype(np.float32))
        np.save(save_path+"vp{}.npy".format(i), vp.cpu().numpy().astype(np.float32))
        np.save(save_path+"srcx{}.npy".format(i), srcxs.float())
        np.save(save_path+"recx{}.npy".format(i), recxs.float())
        i += 1

    del vp, vs, rho, out, vx, vz
    torch.cuda.empty_cache()
    gc.collect()